In [0]:
from pyspark.sql.functions import current_timestamp
def add_ingestion_date(input_df):
  output_df = input_df.withColumn("ingestion_date", current_timestamp())
  return output_df

In [0]:
def merge_delta_data(
    input_df: DataFrame,
    delta_table_path: str,
    merge_condition: str,
    partition_columns: list = None,
):
  table_exists = DeltaTable.isDeltaTable(spark, delta_table_path)
  if table_exists:
      delta_table = DeltaTable.forPath(spark, delta_table_path)
      delta_table = delta_table.alias("tgt") \
        .merge(input_df.alias("src"), merge_condition) \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()
  else:
    if partition_columns:
      input_df.write.mode("overwrite") \
      .format("delta") \
      .partitionBy(*partition_columns) \
      .save(delta_table_path)
    else:
      input_df.write.mode("overwrite") \
      .format("delta") \
      .save(delta_table_path)
